In [ ]:
# Cell 1 — Install & Import Library
!pip install -q tensorflowjs scikit-learn seaborn huggingface_hub

import os, sys, json, time, shutil, random, subprocess, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

def set_global_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Cell 2 — Konfigurasi (path, hyperparameter, daftar eksperimen)

SEED = 42

# ── Sumber Dataset Utama (Kaggle Dataset) & Fallback (HuggingFace) ──
KAGGLE_USERNAME     = os.environ.get("KAGGLE_USERNAME", "")
KAGGLE_DATASET_SLUG = os.environ.get("KAGGLE_DATASET_SLUG", "tokiva-pos-dataset")
KAGGLE_INPUT_DIR    = Path(f"/kaggle/input/{KAGGLE_DATASET_SLUG}")

HF_REPO_ID   = os.environ.get("HF_REPO_ID", "")
HF_REPO_TYPE = "dataset"
HF_TOKEN     = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

# Lokasi fallback download dari HF
DATASET_DIR = Path("/kaggle/working/hf_dataset")
OUTPUT_ROOT = Path("/kaggle/working/Research_POS")

IMAGE_SIZE    = (224, 224)
IMG_HEIGHT, IMG_WIDTH = IMAGE_SIZE
CHANNELS      = 3
NUM_CLASSES_EXPECTED = None  # Dinamis mengikuti jumlah kelas aktual di dataset
BATCH_SIZE    = 32
MAX_EPOCHS    = 50

LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
LABEL_SMOOTHING = 0.1
EMBEDDING_DIM   = 256
MODEL_FAMILY    = "v2"

EARLY_STOPPING_PATIENCE = 7
REDUCE_LR_PATIENCE      = 3
REDUCE_LR_FACTOR        = 0.2
MIN_LR                  = 1e-7
FINE_TUNE_LAST_N        = 15

TRAIN_SIZE      = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE       = 0.15

USE_CACHE    = True
CACHE_IN_RAM = True
CACHE_DIR    = Path("/kaggle/working/research_pos_cache")
if not CACHE_IN_RAM:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

USE_CUTMIX  = False
CUTMIX_ALPHA = 1.0

# Validasi dimensi minimum (menggantikan filter 10KB agar foto WebP 224x224 kecil tetap terbaca)
MIN_IMAGE_DIM = 96
MIN_FILE_SIZE_BYTES = 0

AUTOTUNE = tf.data.AUTOTUNE

# Hanya 2 eksperimen
EXPERIMENTS = {
    "E1_MobileNetV3": {
        "backbone":   "MobileNetV3Large",
        "attention":  None,
    },
    "E2_MobileNetV3_CBAM": {
        "backbone":   "MobileNetV3Large",
        "attention":  "CBAM",
    },
}

for d in ["DATA_REPORT", "LOG", "COMPARISON", "FINAL_MODEL", "EXPERIMENT"]:
    (OUTPUT_ROOT / d).mkdir(parents=True, exist_ok=True)

print("Output:", OUTPUT_ROOT)
print("Experiments:", list(EXPERIMENTS.keys()))


In [ ]:
# Cell 2b — Muat Dataset dari Kaggle Dataset (Utama) atau Fallback dari HuggingFace

def _has_class_folders(root_dir):
    if not root_dir.exists() or not root_dir.is_dir():
        return False
    return any(
        d.is_dir() and not d.name.startswith(".") and d.name.lower() not in {"thumbnails", "data", "dataset"}
        for d in root_dir.iterdir()
    )

def _find_kaggle_input_roots():
    candidates = [KAGGLE_INPUT_DIR]
    if KAGGLE_USERNAME:
        candidates.append(Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{KAGGLE_DATASET_SLUG}"))
    base_input = Path("/kaggle/input")
    if base_input.exists() and base_input.is_dir():
        for p in sorted(base_input.iterdir()):
            if not p.is_dir():
                continue
            if p.name == KAGGLE_DATASET_SLUG:
                candidates.append(p)
            if (p / KAGGLE_DATASET_SLUG).is_dir():
                candidates.append(p / KAGGLE_DATASET_SLUG)
            try:
                for child in p.iterdir():
                    if not child.is_dir():
                        continue
                    if child.name == KAGGLE_DATASET_SLUG:
                        candidates.append(child)
                    elif (child / KAGGLE_DATASET_SLUG).is_dir():
                        candidates.append(child / KAGGLE_DATASET_SLUG)
            except Exception:
                pass
    seen = set()
    for cand in candidates:
        key = str(cand)
        if key in seen or not cand.exists() or not cand.is_dir():
            continue
        seen.add(key)
        kd_data = cand / "data"
        if _has_class_folders(kd_data):
            return [kd_data]
        if _has_class_folders(cand):
            return [cand]
    return []

DATASET_ROOTS = _find_kaggle_input_roots()
DATASET_SOURCE = ""

if DATASET_ROOTS:
    DATASET_SOURCE = f"kaggle:{KAGGLE_DATASET_SLUG}"
    print(f"Menggunakan Kaggle Dataset lokal (instan): {DATASET_ROOTS[0]}")
else:
    DATASET_SOURCE = f"huggingface:{HF_REPO_ID}"
    print(f"Kaggle Input belum tersedia, mengunduh dataset dari HuggingFace: {HF_REPO_ID}")
    hf_root = Path(snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        token=HF_TOKEN if HF_TOKEN else None,
        local_dir=str(DATASET_DIR),
        ignore_patterns=["**/thumbnails/**", "**/.git/**"],
    ))
    roots = [p for p in [hf_root / "data", hf_root / "dataset"] if p.exists() and p.is_dir()]
    DATASET_ROOTS = roots if roots else [hf_root]

DATASET_DIR = DATASET_ROOTS[0]
class_folders = sorted({
    d.name for r in DATASET_ROOTS for d in r.iterdir()
    if d.is_dir() and not d.name.startswith(".") and d.name.lower() not in {"thumbnails", "data", "dataset"}
})
print(f"Sumber dataset: {DATASET_SOURCE} | Jumlah folder kelas: {len(class_folders)}")
for cf in class_folders:
    print(f"  {cf}")


In [ ]:
# Cell 3 — Utility functions (folder, JSON, ukuran file, parameter count)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)

def save_json(data, path):
    path = Path(path)
    ensure_dir(path.parent)
    with open(path, "w") as f:
        json.dump(data, f, indent=4, default=str)

def save_text(text, path):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(str(text), encoding="utf-8")

def now_string():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def file_size_mb(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / (1024**2)
    if path.is_dir():
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / (1024**2)
    return 0.0

def count_parameters(model):
    trainable     = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
    non_trainable = int(np.sum([np.prod(v.shape) for v in model.non_trainable_variables]))
    return {
        "parameters":             trainable + non_trainable,
        "trainable_parameters":   trainable,
        "non_trainable_parameters": non_trainable,
    }

def get_last_4d_layer(model):
    for layer in reversed(model.layers):
        try:
            if len(layer.output.shape) == 4:
                return layer
        except Exception:
            pass
    return None

In [ ]:
# Cell 4 — Kumpulkan file gambar, validasi dimensi PIL, buat label & simpan class.json
#
# FILTER THUMBNAIL & VALIDASI CITRA:
#   - glob("*") -> hanya root class folder, skip subdirectory thumbnails/
#   - Validasi pembacaan gambar & dimensi minimum via PIL (>= 96x96 px) sehingga WebP 224x224 kecil tetap lolos

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def collect_image_files(dataset_roots, min_dim=96):
    roots = dataset_roots if isinstance(dataset_roots, (list, tuple)) else [dataset_roots]
    records = []
    seen = set()
    skipped_invalid = 0
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for class_dir in sorted(root.iterdir()):
            if not class_dir.is_dir():
                continue
            cname = class_dir.name.strip()
            if not cname or cname.startswith(".") or cname.lower() in {"thumbnails", "data", "dataset"}:
                continue
            for img_path in class_dir.glob("*"):
                if not img_path.is_file():
                    continue
                if img_path.suffix.lower() not in VALID_EXTENSIONS:
                    continue
                dedup_key = (cname, img_path.name)
                if dedup_key in seen:
                    continue
                try:
                    with Image.open(img_path) as im:
                        w, h = im.size
                        im.load()
                    if w < min_dim or h < min_dim:
                        skipped_invalid += 1
                        continue
                except Exception:
                    skipped_invalid += 1
                    continue
                seen.add(dedup_key)
                file_size = img_path.stat().st_size
                records.append({
                    "filepath": str(img_path),
                    "class_name": cname,
                    "file_size_kb": round(file_size / 1024, 1),
                })
    df = pd.DataFrame(records)
    if df.empty:
        raise FileNotFoundError(f"Tidak ada gambar valid di {roots}")
    class_names    = sorted(df["class_name"].unique())
    class_to_index = {name: idx for idx, name in enumerate(class_names)}
    df["label"]    = df["class_name"].map(class_to_index)
    print(f"File dikumpulkan: {len(df)}")
    print(f"File di-skip (rusak / < {min_dim}x{min_dim} px): {skipped_invalid}")
    return df, class_names, class_to_index

image_df, CLASS_NAMES, CLASS_TO_INDEX = collect_image_files(DATASET_ROOTS, MIN_IMAGE_DIM)
NUM_CLASSES = len(CLASS_NAMES)

save_json(
    {"class_names": CLASS_NAMES, "class_to_index": CLASS_TO_INDEX,
     "num_classes": NUM_CLASSES, "num_images": len(image_df), "source": DATASET_SOURCE},
    OUTPUT_ROOT / "DATA_REPORT" / "class_mapping.json",
)

class_json = {str(idx): name for idx, name in CLASS_TO_INDEX.items()}
save_json(class_json, OUTPUT_ROOT / "DATA_REPORT" / "class.json")

print(f"Kelas   : {NUM_CLASSES}")
print(f"Gambar  : {len(image_df)}")
if NUM_CLASSES_EXPECTED is not None and NUM_CLASSES != NUM_CLASSES_EXPECTED:
    print(f"PERINGATAN: ekspektasi {NUM_CLASSES_EXPECTED} kelas, aktual {NUM_CLASSES}")
print(CLASS_NAMES)


In [ ]:
# Cell 5 — Stratified split 70/15/15

train_df, tmp_df = train_test_split(
    image_df, test_size=(VALIDATION_SIZE + TEST_SIZE),
    stratify=image_df["label"], random_state=SEED,
)
relative_test = TEST_SIZE / (VALIDATION_SIZE + TEST_SIZE)
validation_df, test_df = train_test_split(
    tmp_df, test_size=relative_test,
    stratify=tmp_df["label"], random_state=SEED,
)

train_df      = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df       = test_df.reset_index(drop=True)

print(f"Train      : {len(train_df)}")
print(f"Validation : {len(validation_df)}")
print(f"Test       : {len(test_df)}")

pd.concat([
    train_df.assign(split="train"),
    validation_df.assign(split="validation"),
    test_df.assign(split="test"),
], ignore_index=True).to_csv(OUTPUT_ROOT / "DATA_REPORT" / "dataset_split.csv", index=False)

In [ ]:
# Cell 6 — Decode, resize (center-crop), augmentasi, CutMix, cache, prefetch

def decode_and_resize(path, label):
    """Resize sisi terpendek ≥ target lalu center-crop — menghindari distorsi dan black bar."""
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])

    shape  = tf.shape(image)
    h, w   = tf.cast(shape[0], tf.float32), tf.cast(shape[1], tf.float32)
    scale  = tf.maximum(IMG_HEIGHT / h, IMG_WIDTH / w)
    new_h  = tf.cast(tf.round(h * scale), tf.int32)
    new_w  = tf.cast(tf.round(w * scale), tf.int32)

    image  = tf.image.resize(image, [new_h, new_w],
                             method=tf.image.ResizeMethod.BILINEAR, antialias=True)
    image  = tf.image.resize_with_crop_or_pad(image, IMG_HEIGHT, IMG_WIDTH)
    image  = tf.cast(image, tf.float32)
    label  = tf.one_hot(label, depth=NUM_CLASSES)
    return image, label

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08, fill_mode="reflect"),
    layers.RandomZoom((-0.10, 0.10), (-0.10, 0.10), fill_mode="reflect"),
    layers.RandomContrast(0.10),
    layers.RandomBrightness(0.10, value_range=(0.0, 255.0)),
], name="data_augmentation")

def cutmix_batch(images, labels, alpha=1.0):
    batch_size      = tf.shape(images)[0]
    idx             = tf.random.shuffle(tf.range(batch_size))
    shuf_images     = tf.gather(images, idx)
    shuf_labels     = tf.gather(labels, idx)
    lam             = tf.cast(tf.compat.v1.distributions.Beta(alpha, alpha).sample(), tf.float32)
    cut_ratio       = tf.sqrt(1.0 - lam)
    cut_h = tf.cast(tf.cast(IMG_HEIGHT, tf.float32) * cut_ratio, tf.int32)
    cut_w = tf.cast(tf.cast(IMG_WIDTH,  tf.float32) * cut_ratio, tf.int32)
    cy = tf.random.uniform([], 0, IMG_HEIGHT, dtype=tf.int32)
    cx = tf.random.uniform([], 0, IMG_WIDTH,  dtype=tf.int32)
    y1 = tf.clip_by_value(cy - cut_h // 2, 0, IMG_HEIGHT)
    y2 = tf.clip_by_value(cy + cut_h // 2, 0, IMG_HEIGHT)
    x1 = tf.clip_by_value(cx - cut_w // 2, 0, IMG_WIDTH)
    x2 = tf.clip_by_value(cx + cut_w // 2, 0, IMG_WIDTH)
    mask = tf.pad(tf.ones((y2 - y1, x2 - x1)),
                  [[y1, IMG_HEIGHT - y2], [x1, IMG_WIDTH - x2]])
    mask = tf.reshape(mask, [IMG_HEIGHT, IMG_WIDTH, 1])
    mixed_images = images * (1 - mask) + shuf_images * mask
    lam_adj = 1.0 - tf.cast((y2 - y1) * (x2 - x1), tf.float32) / tf.cast(IMG_HEIGHT * IMG_WIDTH, tf.float32)
    mixed_labels = lam_adj * labels + (1 - lam_adj) * shuf_labels
    return mixed_images, mixed_labels

def create_dataset(df, split_name, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["filepath"].astype(str).values, df["label"].astype(np.int32).values)
    )
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    if USE_CACHE:
        ds = ds.cache() if CACHE_IN_RAM else ds.cache(str(CACHE_DIR / f"{split_name}.cache"))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    if training and USE_CUTMIX:
        ds = ds.map(lambda x, y: cutmix_batch(x, y, CUTMIX_ALPHA), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds      = create_dataset(train_df,      "train",      training=True)
validation_ds = create_dataset(validation_df, "validation", training=False)
test_ds       = create_dataset(test_df,       "test",       training=False)

print("Dataset siap. CutMix:", USE_CUTMIX, "| Cache:", USE_CACHE)

In [ ]:
# Cell 7 — Implementasi CBAMLayer (satu-satunya attention yang dipakai)

class CBAMLayer(layers.Layer):
    """Convolutional Block Attention Module — channel + spatial attention."""

    def __init__(self, reduction_ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden   = max(channels // self.reduction_ratio, 1)
        self.ch_dense1   = layers.Dense(hidden,   activation="relu", use_bias=True)
        self.ch_dense2   = layers.Dense(channels, activation=None,   use_bias=True)
        self.spatial_conv = layers.Conv2D(1, kernel_size=7, padding="same",
                                          activation="sigmoid", use_bias=False)
        super().build(input_shape)

    def call(self, inputs):
        avg_p = tf.reduce_mean(inputs, axis=[1, 2])
        max_p = tf.reduce_max(inputs,  axis=[1, 2])
        ch_att = tf.nn.sigmoid(
            self.ch_dense2(self.ch_dense1(avg_p)) +
            self.ch_dense2(self.ch_dense1(max_p))
        )
        ch_att = tf.reshape(ch_att, [-1, 1, 1, tf.shape(ch_att)[-1]])
        x = inputs * ch_att
        sp_feat = tf.concat([
            tf.reduce_mean(x, axis=-1, keepdims=True),
            tf.reduce_max(x,  axis=-1, keepdims=True),
        ], axis=-1)
        return x * self.spatial_conv(sp_feat)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"reduction_ratio": self.reduction_ratio})
        return cfg

class L2NormalizeLayer(layers.Layer):
    """Layer normalisasi L2 yang serializable penuh ke SavedModel & TFJS."""

    def __init__(self, axis=-1, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return tf.math.l2_normalize(tf.cast(inputs, tf.float32), axis=self.axis)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"axis": self.axis})
        return cfg


In [ ]:
# Cell 8 — Build MobileNetV3Large ± CBAM + Bottleneck Embedding 256-d (v2 Hybrid), partial fine-tune, compile AdamW

def build_backbone(backbone_name, input_tensor):
    if backbone_name == "MobileNetV3Large":
        return tf.keras.applications.MobileNetV3Large(
            input_tensor=input_tensor,
            include_top=False,
            weights="imagenet",
            include_preprocessing=True,   # PENTING: preprocessing built-in
        )
    raise ValueError(f"Backbone tidak dikenal: {backbone_name}")

def set_partial_trainable(backbone, last_n=15):
    total = len(backbone.layers)
    for i, layer in enumerate(backbone.layers):
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = (i >= total - last_n)

def build_model(backbone_name, attention_name=None):
    inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), dtype=tf.float32)
    backbone = build_backbone(backbone_name, inputs)
    set_partial_trainable(backbone, last_n=FINE_TUNE_LAST_N)
    x = backbone.output
    if attention_name == "CBAM":
        x = CBAMLayer(reduction_ratio=8, name="cbam_attention")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    emb = layers.Dense(EMBEDDING_DIM, use_bias=False, name="embedding_dense")(x)
    emb = layers.BatchNormalization(name="embedding_bn")(emb)
    emb_norm = L2NormalizeLayer(name="embedding")(emb)
    head = layers.Dropout(0.4, name="head_dropout")(emb_norm)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="predictions")(head)
    return keras.Model(
        inputs=inputs,
        outputs=outputs,
        name=f"{backbone_name}_{attention_name or 'Baseline'}_Embed",
    )

def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=min(5, NUM_CLASSES), name="top5_accuracy")],
    )
    return model


In [ ]:
# Cell 9 — EarlyStopping, ReduceLR, ModelCheckpoint, dan fungsi train_experiment

class ResearchLogger(keras.callbacks.Callback):
    def __init__(self, log_path):
        super().__init__()
        self.log_path   = Path(log_path)
        self.start_time = None

    def _write(self, msg):
        with open(self.log_path, "a") as f:
            f.write(f"[{now_string()}] {msg}\n")

    def on_train_begin(self, logs=None):
        self.start_time = time.perf_counter()
        self._write("Start Training")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self._write(f"Epoch {epoch+1} | loss={logs.get('loss'):.4f} | "
                    f"acc={logs.get('accuracy'):.4f} | val_loss={logs.get('val_loss'):.4f} | "
                    f"val_acc={logs.get('val_accuracy'):.4f}")

    def on_train_end(self, logs=None):
        self._write(f"Done — {time.perf_counter() - self.start_time:.1f}s")

def create_experiment_dirs(name):
    exp_dir = OUTPUT_ROOT / "EXPERIMENT" / name
    for sub in ["checkpoint", "history", "model", "saved_model",
                "tensorflowjs", "tflite", "plots", "confusion_matrix",
                "prediction", "report", "weights", "log"]:
        ensure_dir(exp_dir / sub)
    return exp_dir

def create_callbacks(exp_dir):
    return [
        keras.callbacks.ModelCheckpoint(
            str(exp_dir / "checkpoint" / "best.weights.h5"),
            monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=REDUCE_LR_FACTOR,
            patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1),
        ResearchLogger(exp_dir / "log" / "training.txt"),
    ]

def train_experiment(exp_name, exp_config):
    exp_dir = create_experiment_dirs(exp_name)
    model   = compile_model(build_model(exp_config["backbone"], exp_config["attention"]))

    print(f"\n{'='*70}\n{exp_name}\n{'='*70}")
    param_info = count_parameters(model)
    display(pd.DataFrame(param_info.items(), columns=["metric", "value"]))

    t0 = time.perf_counter()
    history = model.fit(
        train_ds, validation_data=validation_ds,
        epochs=MAX_EPOCHS, callbacks=create_callbacks(exp_dir), verbose=1,
    )
    train_time = time.perf_counter() - t0

    hist_df   = pd.DataFrame(history.history)
    best_epoch = int(hist_df["val_loss"].idxmin() + 1)
    hist_df.to_csv(exp_dir / "history" / "history.csv", index=False)

    # Plot training curve
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(hist_df["accuracy"],     label="Train");  axes[0].plot(hist_df["val_accuracy"], label="Val")
    axes[0].set_title(f"{exp_name} — Accuracy"); axes[0].legend()
    axes[1].plot(hist_df["loss"],         label="Train");  axes[1].plot(hist_df["val_loss"],     label="Val")
    axes[1].set_title(f"{exp_name} — Loss");     axes[1].legend()
    plt.tight_layout()
    plt.savefig(exp_dir / "plots" / "training_history.png", dpi=200)
    plt.show()

    model.save(exp_dir / "model" / "model.keras")
    model.save_weights(exp_dir / "weights" / "final.weights.h5")

    training_info = {
        **param_info,
        "training_time_seconds": train_time,
        "best_epoch": best_epoch,
        "best_val_accuracy": float(hist_df.loc[best_epoch - 1, "val_accuracy"]),
        "best_val_loss":     float(hist_df.loc[best_epoch - 1, "val_loss"]),
    }
    save_json(training_info, exp_dir / "report" / "training_info.json")

    return {"model": model, "history": hist_df,
            "experiment_dir": exp_dir, "training_info": training_info}

In [ ]:
# Cell 10 — Training E1_MobileNetV3 dan E2_MobileNetV3_CBAM

RESULTS = {}

for exp_name, exp_config in EXPERIMENTS.items():
    set_global_seed(SEED)
    RESULTS[exp_name] = train_experiment(exp_name, exp_config)

print("\nSeluruh eksperimen selesai.")

In [ ]:
# Cell 11 — Prediksi test set, hitung metrik, confusion matrix, classification report
#          + simpan kesalahan model (misclassified.csv + error_summary.csv)

def predict_dataset(model, dataset):
    y_true, y_prob = [], []
    for images, labels in dataset:
        y_prob.append(model.predict(images, verbose=0))
        y_true.append(labels.numpy() if hasattr(labels, "numpy") else np.asarray(labels))
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    return np.argmax(y_true, 1), np.argmax(y_prob, 1), y_true, y_prob

def measure_inference(model, dataset, n_warmup=3):
    batch_images = next(iter(dataset))[0]
    for _ in range(n_warmup):
        model(batch_images, training=False)
    t0, total = time.perf_counter(), 0
    for images, _ in dataset:
        model(images, training=False)
        total += int(images.shape[0])
    elapsed = max(time.perf_counter() - t0, 1e-9)

    # Single-image inference (batch=1) agar lebih mencerminkan prediksi 1 frame
    single_samples = []
    for images, _ in dataset:
        for i in range(int(images.shape[0])):
            single_samples.append(tf.expand_dims(images[i], axis=0))
            if len(single_samples) >= 50:
                break
        if len(single_samples) >= 50:
            break

    single_ms = None
    single_fps = None
    if single_samples:
        for _ in range(5):
            model(single_samples[0], training=False)
        ts0 = time.perf_counter()
        for sample in single_samples:
            model(sample, training=False)
        elapsed_single = max(time.perf_counter() - ts0, 1e-9)
        single_ms = elapsed_single / len(single_samples) * 1000.0
        single_fps = len(single_samples) / elapsed_single

    return {
        "inference_time_per_image_ms": elapsed / total * 1000.0,
        "fps": total / elapsed,
        "inference_single_image_ms": single_ms,
        "fps_single_image": single_fps,
    }

def compute_topk_accuracy(y_true_ids, y_prob, k=5):
    k_eff = min(int(k), y_prob.shape[1])
    if k_eff <= 0 or len(y_true_ids) == 0:
        return 0.0
    top_k_preds = np.argsort(y_prob, axis=1)[:, -k_eff:]
    hits = [1 if y_true_ids[i] in top_k_preds[i] else 0 for i in range(len(y_true_ids))]
    return float(np.mean(hits))

def save_misclassified(y_true_ids, y_pred_ids, y_prob, test_df, exp_dir):
    """Simpan detail kesalahan model: true label, predicted label, confidence."""
    misclassified_idx = np.where(y_true_ids != y_pred_ids)[0]
    if len(misclassified_idx) == 0:
        print("  Tidak ada kesalahan klasifikasi!")
        empty_mis = pd.DataFrame(columns=[
            "filepath", "true_label", "predicted_label", "confidence", "true_confidence"
        ])
        empty_err = pd.DataFrame(columns=[
            "true_label", "total_errors", "most_confused_with", "most_confused_count", "avg_confidence"
        ])
        empty_mis.to_csv(exp_dir / "report" / "misclassified.csv", index=False)
        empty_err.to_csv(exp_dir / "confusion_matrix" / "error_summary.csv", index=False)
        return empty_mis

    rows = []
    for idx in misclassified_idx:
        true_label = CLASS_NAMES[y_true_ids[idx]]
        pred_label = CLASS_NAMES[y_pred_ids[idx]]
        confidence = float(y_prob[idx, y_pred_ids[idx]])
        true_conf  = float(y_prob[idx, y_true_ids[idx]])
        rows.append({
            "filepath":        test_df.iloc[idx]["filepath"],
            "true_label":      true_label,
            "predicted_label": pred_label,
            "confidence":      round(confidence, 4),
            "true_confidence": round(true_conf, 4),
        })
    mis_df = pd.DataFrame(rows)
    mis_df.to_csv(exp_dir / "report" / "misclassified.csv", index=False)

    # Error summary: per class — total error, most confused with
    error_rows = []
    for true_cls in CLASS_NAMES:
        cls_errors = mis_df[mis_df["true_label"] == true_cls]
        if len(cls_errors) == 0:
            continue
        most_confused = cls_errors["predicted_label"].value_counts().idxmax()
        most_confused_count = int(cls_errors["predicted_label"].value_counts().max())
        avg_confidence = float(cls_errors["confidence"].mean())
        error_rows.append({
            "true_label":          true_cls,
            "total_errors":        len(cls_errors),
            "most_confused_with":  most_confused,
            "most_confused_count": most_confused_count,
            "avg_confidence":      round(avg_confidence, 4),
        })
    error_summary_df = pd.DataFrame(error_rows).sort_values("total_errors", ascending=False)
    error_summary_df.to_csv(exp_dir / "confusion_matrix" / "error_summary.csv", index=False)

    print(f"  Total misclassified: {len(mis_df)}")
    print(f"  Classes with errors: {len(error_summary_df)}")
    return mis_df

def evaluate_experiment(exp_name, result):
    model   = result["model"]
    exp_dir = result["experiment_dir"]

    y_true_ids, y_pred_ids, y_true_oh, y_prob = predict_dataset(model, test_ds)

    acc  = accuracy_score(y_true_ids, y_pred_ids)
    prec = precision_score(y_true_ids, y_pred_ids, average="macro", zero_division=0)
    rec  = recall_score(y_true_ids, y_pred_ids, average="macro", zero_division=0)
    f1   = f1_score(y_true_ids, y_pred_ids, average="macro", zero_division=0)
    top5 = compute_topk_accuracy(y_true_ids, y_prob, k=5)
    try:
        auc = roc_auc_score(y_true_oh, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = np.nan

    num_misclassified = int(np.sum(y_true_ids != y_pred_ids))
    num_test_images   = int(len(y_true_ids))

    inf_info = measure_inference(model, test_ds)
    metrics  = {
        "experiment":       exp_name,
        "accuracy":         acc,
        "precision_macro":  prec,
        "recall_macro":     rec,
        "f1_macro":         f1,
        "top5_accuracy":    top5,
        "auc_macro_ovr":    auc,
        "num_misclassified": num_misclassified,
        "num_test_images":   num_test_images,
        **inf_info,
        **count_parameters(model),
        "training_time_seconds": result["training_info"]["training_time_seconds"],
        "best_epoch":            result["training_info"]["best_epoch"],
        "best_val_accuracy":     result["training_info"]["best_val_accuracy"],
    }
    pd.DataFrame([metrics]).to_csv(exp_dir / "metrics.csv", index=False)

    # Classification Report (sertakan index_label="class_name" agar terbaca rapi)
    report = classification_report(
        y_true_ids, y_pred_ids, target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    pd.DataFrame(report).T.to_csv(
        exp_dir / "report" / "classification_report.csv",
        index_label="class_name",
    )

    # Confusion Matrix
    cm = confusion_matrix(y_true_ids, y_pred_ids, labels=np.arange(NUM_CLASSES))
    plt.figure(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix — {exp_name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.xticks(rotation=90); plt.tight_layout()
    plt.savefig(exp_dir / "confusion_matrix" / "confusion_matrix.png", dpi=200)
    plt.show()

    # Simpan kesalahan model
    print(f"\nKesalahan model — {exp_name}:")
    save_misclassified(y_true_ids, y_pred_ids, y_prob, test_df, exp_dir)

    result.update({
        "metrics": metrics,
        "y_true_ids": y_true_ids, "y_pred_ids": y_pred_ids,
        "y_true_onehot": y_true_oh, "y_prob": y_prob,
    })
    return result

for exp_name in RESULTS:
    RESULTS[exp_name] = evaluate_experiment(exp_name, RESULTS[exp_name])

print("\nEvaluasi selesai.")

In [ ]:
# Cell 12 — Export Dual-Output [predictions, embedding] ke SavedModel & TFJS + Hitung Class Centroids (v2)

def build_dual_output_model(trained_model):
    pred_tensor = trained_model.get_layer("predictions").output
    emb_tensor  = trained_model.get_layer("embedding").output
    dual_model  = keras.Model(
        inputs=trained_model.input,
        outputs=[pred_tensor, emb_tensor],
        name=f"{trained_model.name}_DualOutput",
    )
    return dual_model

def compute_class_centroids_and_stats(dual_model, datasets):
    all_vecs = []
    all_labels = []
    for ds in datasets:
        if ds is None:
            continue
        for images, labels in ds:
            _, embs = dual_model(images, training=False)
            embs_np = embs.numpy() if hasattr(embs, "numpy") else np.asarray(embs)
            lbls_np = labels.numpy() if hasattr(labels, "numpy") else np.asarray(labels)
            all_vecs.append(embs_np)
            all_labels.append(np.argmax(lbls_np, axis=1))

    if not all_vecs:
        return {"dimensi": int(EMBEDDING_DIM), "num_classes": 0, "classes": {}}, {
            "intra_class_mean_sim": None,
            "inter_class_mean_sim": None,
        }

    vecs = np.concatenate(all_vecs, axis=0)
    lbls = np.concatenate(all_labels, axis=0)

    classes_meta = {}
    centroids_matrix = []
    intra_sims = []

    for cls_idx in sorted(set(int(x) for x in lbls)):
        mask = (lbls == cls_idx)
        cls_vecs = vecs[mask]
        mean_vec = np.mean(cls_vecs, axis=0)
        norm = float(np.linalg.norm(mean_vec)) + 1e-12
        centroid = (mean_vec / norm).astype(np.float32)
        sims = np.dot(cls_vecs, centroid)
        cls_name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f"class_{cls_idx}"
        classes_meta[cls_name] = {
            "class_index": int(cls_idx),
            "count": int(cls_vecs.shape[0]),
            "mean_sim": round(float(np.mean(sims)), 4),
            "min_sim": round(float(np.min(sims)), 4),
            "centroid": [round(float(v), 6) for v in centroid.tolist()],
        }
        centroids_matrix.append(centroid)
        intra_sims.append(float(np.mean(sims)))

    inter_sim_mean = None
    if len(centroids_matrix) > 1:
        cmat = np.stack(centroids_matrix, axis=0)
        sim_mat = np.dot(cmat, cmat.T)
        tri = sim_mat[np.triu_indices(len(centroids_matrix), k=1)]
        inter_sim_mean = round(float(np.mean(tri)), 4)

    stats = {
        "intra_class_mean_sim": round(float(np.mean(intra_sims)), 4) if intra_sims else None,
        "inter_class_mean_sim": inter_sim_mean,
    }
    return {"dimensi": int(EMBEDDING_DIM), "num_classes": len(classes_meta), "classes": classes_meta}, stats

def export_saved_model(model, out_dir):
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    try:
        model.export(str(out_dir))
    except Exception:
        tf.saved_model.save(model, str(out_dir))
    return out_dir

def export_tflite(model, out_path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = []
    tflite_bytes = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_bytes)
    return out_path

def export_tensorflowjs(saved_model_dir, tfjs_dir):
    tfjs_dir = Path(tfjs_dir)
    if tfjs_dir.exists():
        shutil.rmtree(tfjs_dir)
    ensure_dir(tfjs_dir)
    cmd = [sys.executable, "-m", "tensorflowjs.converters.converter",
           "--input_format=tf_saved_model", str(saved_model_dir), str(tfjs_dir)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        res = subprocess.run(
            ["tensorflowjs_converter", "--input_format=tf_saved_model",
             str(saved_model_dir), str(tfjs_dir)],
            capture_output=True, text=True,
        )
    save_text(res.stdout + "\n" + res.stderr, tfjs_dir / "conversion_log.txt")
    if res.returncode == 0:
        print(f"TFJS Dual-Output OK -> {tfjs_dir}")
    else:
        print(f"TFJS GAGAL: {res.stderr[:300]}")
    return tfjs_dir

def export_all_formats(exp_name, result):
    model   = result["model"]
    exp_dir = result["experiment_dir"]
    dual_model = build_dual_output_model(model)

    saved_model_dir = exp_dir / "saved_model" / "model"
    tflite_path     = exp_dir / "tflite" / "model.tflite"
    tfjs_dir        = exp_dir / "tensorflowjs"

    export_saved_model(dual_model, saved_model_dir)
    export_tflite(model, tflite_path)
    export_tensorflowjs(saved_model_dir, tfjs_dir)

    centroids_ds = [globals().get("train_eval_ds") or globals().get("train_ds"), globals().get("validation_ds")]
    centroids_payload, emb_stats = compute_class_centroids_and_stats(dual_model, centroids_ds)
    save_json(centroids_payload, tfjs_dir / "class_centroids.json")
    emb_meta = {
        "model_family": MODEL_FAMILY,
        "experiment": exp_name,
        "dimensi_embedding": int(EMBEDDING_DIM),
        "output_order": ["predictions", "embedding"],
        "mendukung_embedding": True,
        **emb_stats,
    }
    save_json(emb_meta, tfjs_dir / "embedding_meta.json")
    result["embedding_stats"] = emb_stats

    export_info = {
        "saved_model_size_mb":  file_size_mb(saved_model_dir),
        "tflite_size_mb":       file_size_mb(tflite_path),
        "tensorflowjs_size_mb": file_size_mb(tfjs_dir),
    }
    save_json(export_info, exp_dir / "report" / "export_info.json")
    result["export_info"] = export_info
    return result

for exp_name in RESULTS:
    print(f"\nExporting v2 Dual-Output {exp_name}...")
    RESULTS[exp_name] = export_all_formats(exp_name, RESULTS[exp_name])

print("\nSemua export v2 selesai.")


In [ ]:
# Cell 13 — Buat tabel perbandingan 2 model, pilih yang terbaik berdasarkan accuracy -> f1 -> inference

rows = []
for exp_name, result in RESULTS.items():
    row = result["metrics"].copy()
    ei  = result.get("export_info", {})
    row.update({
        "model_size_mb_keras":        file_size_mb(result["experiment_dir"] / "model" / "model.keras"),
        "model_size_mb_tflite":       ei.get("tflite_size_mb", np.nan),
        "model_size_mb_tensorflowjs": ei.get("tensorflowjs_size_mb", np.nan),
    })
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(OUTPUT_ROOT / "COMPARISON" / "comparison_table.csv", index=False)

ranking_df = comparison_df.sort_values(
    by=["accuracy", "f1_macro", "inference_time_per_image_ms"],
    ascending=[False, False, True],
).reset_index(drop=True)
ranking_df["rank"] = np.arange(1, len(ranking_df) + 1)
ranking_df.to_csv(OUTPUT_ROOT / "COMPARISON" / "ranking.csv", index=False)

best_experiment_name = ranking_df.iloc[0]["experiment"]

save_json({
    "best_experiment": best_experiment_name,
    "num_classes": NUM_CLASSES,
    "num_test_images": int(len(test_df)),
    "generated_at": now_string(),
    "comparison": comparison_df.replace({np.nan: None}).to_dict("records"),
    "ranking": ranking_df.replace({np.nan: None}).to_dict("records"),
}, OUTPUT_ROOT / "COMPARISON" / "research_comparison.json")

print(f"\n{'='*60}")
print(f"Model terbaik: {best_experiment_name}")
print(f"{'='*60}")
display(ranking_df[["rank", "experiment", "accuracy", "precision_macro", "recall_macro",
                     "f1_macro", "num_misclassified", "num_test_images",
                     "inference_time_per_image_ms", "inference_single_image_ms",
                     "model_size_mb_tensorflowjs"]])

In [ ]:
# Cell 14 — Salin model terbaik (TFJS + TFLite + Keras) ke folder FINAL_MODEL beserta class.json
#          + salin misclassified.csv + error_summary.csv + metrics + training_info

best_result  = RESULTS[best_experiment_name]
FINAL_DIR    = OUTPUT_ROOT / "FINAL_MODEL"

if FINAL_DIR.exists():
    shutil.rmtree(FINAL_DIR)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

for folder in ["model", "saved_model", "tensorflowjs", "tflite", "weights", "report", "confusion_matrix"]:
    src = best_result["experiment_dir"] / folder
    if src.exists():
        shutil.copytree(src, FINAL_DIR / folder)

shutil.copy(best_result["experiment_dir"] / "metrics.csv", FINAL_DIR / "metrics.csv")

# Salin class.json ke dalam folder TFJS agar siap dipakai langsung di web
shutil.copy(OUTPUT_ROOT / "DATA_REPORT" / "class.json",
            FINAL_DIR / "tensorflowjs" / "class.json")

# ── Ringkasan lengkap untuk halaman admin ──────────────────────
# Mengumpulkan: metrik, training info, kesalahan model, error summary → 1 JSON
best_metrics = best_result["metrics"]
best_train   = best_result["training_info"]

# Baca misclassified + error_summary
misclassified_path = best_result["experiment_dir"] / "report" / "misclassified.csv"
error_summary_path = best_result["experiment_dir"] / "confusion_matrix" / "error_summary.csv"

misclassified_list = []
if misclassified_path.exists():
    misclassified_list = pd.read_csv(misclassified_path).to_dict("records")

error_summary_list = []
if error_summary_path.exists():
    error_summary_list = pd.read_csv(error_summary_path).to_dict("records")

# Baca history training
history_path = best_result["experiment_dir"] / "history" / "history.csv"
history_list = []
if history_path.exists():
    history_list = pd.read_csv(history_path).to_dict("records")

# Baca classification report
cls_report_path = best_result["experiment_dir"] / "report" / "classification_report.csv"
cls_report_dict = {}
if cls_report_path.exists():
    cls_report_df = pd.read_csv(cls_report_path, index_col=0)
    cls_report_dict = cls_report_df.to_dict("index")

training_summary = {
    "model_family":       MODEL_FAMILY,
    "mendukung_embedding": True,
    "dimensi_embedding":  int(EMBEDDING_DIM),
    "embedding_stats":    best_result.get("embedding_stats", {}),
    "best_experiment":    best_experiment_name,
    "selection_rule":     ["accuracy desc", "f1_macro desc", "inference_time_per_image_ms asc"],
    "accuracy":           float(best_metrics["accuracy"]),
    "f1_macro":           float(best_metrics["f1_macro"]),
    "precision_macro":    float(best_metrics["precision_macro"]),
    "recall_macro":       float(best_metrics["recall_macro"]),
    "auc_macro_ovr":      float(best_metrics["auc_macro_ovr"]) if not np.isnan(best_metrics["auc_macro_ovr"]) else None,
    "top5_accuracy":      float(best_metrics["top5_accuracy"]) if best_metrics.get("top5_accuracy") is not None else None,
    "inference_ms":       float(best_metrics["inference_time_per_image_ms"]),
    "inference_single_ms": float(best_metrics["inference_single_image_ms"]) if best_metrics.get("inference_single_image_ms") is not None else None,
    "fps":                float(best_metrics["fps"]),
    "parameters":         int(best_metrics["parameters"]),
    "best_epoch":         int(best_metrics["best_epoch"]),
    "best_val_accuracy":  float(best_metrics["best_val_accuracy"]),
    "training_time_s":    float(best_metrics["training_time_seconds"]),
    "num_classes":        NUM_CLASSES,
    "num_test_images":    len(test_df),
    "num_misclassified":  len(misclassified_list),
    "training_history":   history_list,
    "classification_report": cls_report_dict,
    "error_summary":      error_summary_list,
    "misclassified_samples": misclassified_list[:50],  # top 50 saja
    "generated_at":       now_string(),
}

save_json(training_summary, FINAL_DIR / "training_summary.json")
save_json(training_summary, FINAL_DIR / "tensorflowjs" / "training_summary.json")

save_json({
    "best_experiment":  best_experiment_name,
    "selection_rule":   ["accuracy desc", "f1_macro desc", "inference_time_per_image_ms asc"],
    "accuracy":         best_result["metrics"]["accuracy"],
    "f1_macro":         best_result["metrics"]["f1_macro"],
}, FINAL_DIR / "best_model_info.json")

print(f"FINAL_MODEL siap di: {FINAL_DIR}")
print(f"Isi FINAL_MODEL/tensorflowjs:")
for f in sorted((FINAL_DIR / "tensorflowjs").iterdir()):
    print(f"  {f.name}  ({file_size_mb(f):.2f} MB)")
print(f"\ntraining_summary.json siap untuk halaman admin.")
print(f"  Misclassified: {len(misclassified_list)} gambar")
print(f"  Error summary: {len(error_summary_list)} class dengan error")

In [ ]:
# Cell 16 — Ringkasan akurasi evaluasi kedua model dalam DataFrame

metrics_cols = [
    "experiment",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "auc_macro_ovr",
    "top5_accuracy",
    "inference_time_per_image_ms",
    "fps",
    "parameters",
    "best_epoch",
    "best_val_accuracy",
    "training_time_seconds",
]

eval_rows = []
for exp_name, result in RESULTS.items():
    m = result["metrics"]
    eval_rows.append({col: m.get(col, np.nan) for col in metrics_cols})

eval_df = pd.DataFrame(eval_rows)

# Format angka biar rapi
format_map = {
    "accuracy":                    "{:.4f}",
    "precision_macro":             "{:.4f}",
    "recall_macro":                "{:.4f}",
    "f1_macro":                    "{:.4f}",
    "auc_macro_ovr":               "{:.4f}",
    "top5_accuracy":               "{:.4f}",
    "inference_time_per_image_ms": "{:.4f}",
    "fps":                         "{:.2f}",
    "best_val_accuracy":           "{:.4f}",
    "training_time_seconds":       "{:.1f}",
}

eval_display = eval_df.copy()
for col, fmt in format_map.items():
    if col in eval_display.columns:
        eval_display[col] = eval_display[col].apply(lambda x: fmt.format(x) if pd.notna(x) else "-")

display(eval_display)

# Highlight model terbaik
best = eval_df.loc[eval_df["accuracy"].idxmax(), "experiment"]
best_acc = eval_df["accuracy"].max()
print(f"\nModel terbaik : {best}")
print(f"Accuracy      : {best_acc:.4f} ({best_acc*100:.2f}%)")

In [ ]:
# Cell 17 — ZIP isi TFJS best model (model.json + *.bin + class.json), nama ZIP = nama model

import os
from IPython.display import FileLink, display as ipy_display

TFJS_DIR = FINAL_DIR / "tensorflowjs"
ZIP_PATH = str(OUTPUT_ROOT / f"{best_experiment_name}.zip")

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# Hanya ambil file relevan saja, exclude conversion_log.txt
include_files = (
    list(TFJS_DIR.glob("model.json")) +
    list(TFJS_DIR.glob("*.bin")) +
    [TFJS_DIR / "class.json", TFJS_DIR / "class_centroids.json", TFJS_DIR / "embedding_meta.json"]
)

# Pastikan class.json ada
if not (TFJS_DIR / "class.json").exists():
    shutil.copy(OUTPUT_ROOT / "DATA_REPORT" / "class.json", TFJS_DIR / "class.json")

file_list = " ".join(f'"{f.name}"' for f in include_files if f.exists())
os.system(f'cd "{TFJS_DIR}" && zip -q "{ZIP_PATH}" {file_list}')

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"ZIP     : {best_experiment_name}.zip")
print(f"Ukuran  : {size_mb:.2f} MB")
print(f"Isi ZIP :")
for f in include_files:
    if f.exists():
        print(f"  ✓ {f.name}  ({file_size_mb(f):.3f} MB)")

ipy_display(FileLink(ZIP_PATH))